# REBUILD EVERYTHING on Kaggle (cache + models + results), one notebook
Use this only because the cache / checkpoints are lost. Expected time: about 4 to 7 hours on a GPU.

**Before running:** right panel -> Session options: Accelerator = **GPU (T4 or P100)**, Internet = **On**. Add Input -> your Dataset that contains `iwildcam_adapt_FINAL.zip`.

**Run it without keeping the tab open:** top right **Save Version -> Save & Run All (Commit)**. It keeps running on Kaggle (max 12 h). When it finishes, open the version's **Output** tab and download `outputs.zip`.

Check in the log: every step prints a line starting with `### STEP`. If one fails, the notebook stops there and shows the error.

In [ ]:
# 0. Code + GPU + packages
import os, glob, shutil, zipfile, subprocess, json
def sh(cmd):
    print('$', cmd, flush=True)
    r = os.system(cmd)
    assert r == 0, f'command failed ({r}): {cmd}'
sh('nvidia-smi | head -12')
hits = glob.glob('/kaggle/input/**/src/run_adaptation.py', recursive=True)
if hits:
    CODE = os.path.dirname(os.path.dirname(hits[0]))
else:
    zips = glob.glob('/kaggle/input/**/*.zip', recursive=True)
    assert zips, 'Add your code dataset: right panel -> Add Input -> Your datasets'
    zipfile.ZipFile(zips[0]).extractall('/kaggle/working/_code')
    CODE = os.path.dirname(os.path.dirname(glob.glob('/kaggle/working/_code/**/src/run_adaptation.py', recursive=True)[0]))
W = '/kaggle/working/iwildcam_adapt'
shutil.rmtree(W, ignore_errors=True)
shutil.copytree(CODE, W, copy_function=shutil.copyfile)
os.system(f'chmod -R u+w {W}')
os.chdir(W)
sh('pip install -q wilds timm gradio scikit-learn tqdm huggingface_hub')
DATA  = '/tmp/wilds_data'                 # raw dataset, ~25 GB, lost after the session (fine)
CACHE = '/tmp/iwildcam_cache'             # 224px subset
CKPT, RES = '/kaggle/working/outputs/ckpt', '/kaggle/working/outputs/results'
os.makedirs(CKPT, exist_ok=True); os.makedirs(RES, exist_ok=True)
SEEDS = '0 1 2'          # use '0 1 2 3 4' for 5 seeds if you have time (slower)
import torch; assert torch.cuda.is_available(), 'No GPU: turn on Accelerator = GPU in Session options'
print('GPU OK:', torch.cuda.get_device_name(0))

In [ ]:
print('### STEP 1: download iWildCam (Hugging Face mirror, ~12 GB, 20-40 min)')
sh(f'python scripts/download_iwildcam_hf.py --root {DATA}')

In [ ]:
print('### STEP 2: build the 224px subset cache (1-3 h)')
sh(f'python -m src.build_cache --root {DATA} --cache-dir {CACHE} --train-cap-per-class 100 --per-camera-cap 400 --min-camera-images 150 --workers 4 --no-download')
sh(f'cp {CACHE}/class_names.json /kaggle/working/outputs/class_names.json')
sh(f'ls -la {CACHE} | head -30')

In [ ]:
print('### STEP 3: train both backbones')
for bb in ['resnet50', 'dino_vits16']:
    sh(f'python -m src.train --backbone {bb} --cache-dir {CACHE} --out {CKPT} --epochs 5')
sh(f'ls -la {CKPT}')

In [ ]:
print('### STEP 4: baseline (seen vs unseen) and tuning on val cameras')
sh(f'python -m src.baseline_eval --cache-dir {CACHE} --ckpt-dir {CKPT} --out {RES}')
sh(f'python -m src.tune --cache-dir {CACHE} --ckpt-dir {CKPT} --out {RES}')
print(open(f'{RES}/baseline.json').read())

In [ ]:
print('### STEP 5: MAIN experiment (n = 16/32/64/128, plus oracle)')
sh(f'python -m src.run_adaptation --cache-dir {CACHE} --ckpt-dir {CKPT} --out {RES} --methods bn tent --sizes 16 32 64 128 --seeds {SEEDS} --oracle')

In [ ]:
print('### STEP 6: ablations (last layers only; share of empty frames)')
sh(f'python -m src.run_adaptation --cache-dir {CACHE} --ckpt-dir {CKPT} --out {RES} --methods bn tent --sizes 64 --seeds 0 1 2 --scope last')
for ef in [0.0, 0.25, 0.5, 0.75]:
    sh(f'python -m src.run_adaptation --cache-dir {CACHE} --ckpt-dir {CKPT} --out {RES} --methods bn tent --sizes 64 --seeds 0 1 2 --empty-frac {ef}')

In [ ]:
print('### STEP 7: tables + figures, then pack everything')
sh(f'python -m src.analysis --results {RES} --cache-dir {CACHE} --focus-n 64')
print(open(f'{RES}/summary.md').read())
sh('cd /kaggle/working && zip -qr outputs.zip outputs && ls -la outputs.zip')
print('DONE. Download outputs.zip from the Output tab (also contains ckpt/ with the models and class_names.json for the demo).')